In [0]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
 
CATALOG   = "clutchlytics"
SCHEMA    = "bronze"
VOLUME    = "nhl_goalie_logs"
TABLE     = f"{CATALOG}.{SCHEMA}.raw_nhl_goalie_logs"
 
# Hardcoded — meta.season = 1 due to pull script bug, correct value is 2026
SEASON    = 2026
 
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
 
print(f"Source : {VOLUME_PATH}")
print(f"Target : {TABLE}")
print(f"Season : {SEASON} (hardcoded — meta.season bug in pull script)")

In [0]:
# ── DISCOVER FILES ────────────────────────────────────────────────────────────
 
all_files    = dbutils.fs.ls(VOLUME_PATH)
goalie_files = [f for f in all_files if f.name.endswith(".json")]
 
print(f"Goalie files found: {len(goalie_files)}")
for f in sorted(goalie_files, key=lambda x: x.name):
    print(f"  {f.name:<45} {f.size:>8,} bytes")

In [0]:
# ── READ + PARSE ALL FILES ────────────────────────────────────────────────────
 
import json
from datetime import datetime, timezone
 
ingested_at = datetime.now(timezone.utc).isoformat()
rows        = []
skipped     = []
 
for f in goalie_files:
    filename = f.name  # e.g. 4712036_jeremy_swayman.json
 
    # ── Parse athlete_id from filename ──
    try:
        athlete_id = filename.split("_")[0]
        if not athlete_id.isdigit():
            raise ValueError(f"First segment not numeric: {athlete_id}")
    except Exception as e:
        skipped.append((filename, str(e)))
        continue
 
    # ── Read and parse JSON ──
    try:
        raw_text = spark.read.text(f.path)
        json_str = "\n".join([row.value for row in raw_text.collect()])
        payload  = json.loads(json_str)
    except Exception as e:
        skipped.append((filename, f"parse error: {e}"))
        continue
 
    # ── Extract meta envelope ──
    meta              = payload.get("meta", {})
    pulled_at         = meta.get("pulled_at")
    athlete_name      = meta.get("athlete_name")
    team_id           = meta.get("team_id")
    team_abbreviation = meta.get("team_abbreviation")
    season_type       = meta.get("season_type")
    round_num         = meta.get("round")
    source_url        = meta.get("source_url")
    # Note: meta.season intentionally ignored — bug in pull script returns 1
    # SEASON constant (2026) used instead
 
    # ── Extract data block ──
    data          = payload.get("data", {})
    labels        = data.get("labels", [])
    names         = data.get("names", [])
    display_names = data.get("displayNames", [])
    season_types  = data.get("seasonTypes", [])
 
    # ── Count game rows across all season types ──
    total_games   = 0
    playoff_games = 0
    regular_games = 0
 
    for st in season_types:
        for cat in st.get("categories", []):
            for event in cat.get("events", []):
                total_games += 1
                type_abbr = event.get("type", {}).get("abbreviation", "")
                if type_abbr.startswith("RD") or "round" in event.get("type", {}).get("slug", ""):
                    playoff_games += 1
                else:
                    regular_games += 1
 
    rows.append({
        # ── Athlete identity (from meta) ──
        "athlete_id":         athlete_id,
        "athlete_name":       athlete_name,
        "team_id":            team_id,
        "team_abbreviation":  team_abbreviation,
 
        # ── Stat schema — pipe-delimited strings ──
        # Silver uses these to zip with stats[] arrays
        "labels":             "|".join(labels),
        "names":              "|".join(names),
        "display_names":      "|".join(display_names),
 
        # ── Raw gamelog — full seasonTypes block as JSON string ──
        "season_types_json":  json.dumps(season_types),
 
        # ── Game count summary (for validation) ──
        "total_game_rows":    total_games,
        "playoff_game_rows":  playoff_games,
        "regular_game_rows":  regular_games,
 
        # ── Season / context ──
        "season":             SEASON,
        "season_type":        season_type,
        "round":              round_num,
 
        # ── Ingestion metadata ──
        "source_file":        filename,
        "source_url":         source_url,
        "pulled_at":          pulled_at,
        "ingested_at":        ingested_at,
    })
 
    print(f"  {filename:<45} → {total_games:>3} games "
          f"({playoff_games} playoff, {regular_games} regular)")
 
print(f"\nRows built  : {len(rows)}")
print(f"Skipped     : {len(skipped)}")
 
if skipped:
    print("\nSkipped files:")
    for fname, reason in skipped:
        print(f"  {fname}: {reason}")

In [0]:
# ── SPOT CHECK — verify label/stats zip on one goalie ─────────────────────────
 
sample = rows[0] if rows else None
 
if sample:
    names_list  = sample["names"].split("|")
    season_data = json.loads(sample["season_types_json"])
 
    first_event = None
    for st in season_data:
        for cat in st.get("categories", []):
            events = cat.get("events", [])
            if events:
                first_event = events[0]
                break
        if first_event:
            break
 
    if first_event:
        stats  = first_event.get("stats", [])
        zipped = dict(zip(names_list, stats))
        print(f"Goalie     : {sample['athlete_name']} ({sample['athlete_id']})")
        print(f"Event ID   : {first_event.get('eventId')}")
        print(f"Game type  : {first_event.get('type', {}).get('text')}")
        print(f"\nLabel → stat zip:")
        for k, v in zipped.items():
            print(f"  {k:<25} = {v}")

In [0]:
# ── WRITE TO DELTA ────────────────────────────────────────────────────────────
# MERGE on athlete_id — Option A (overwrite with latest snapshot, no history).
 
if not rows:
    raise ValueError("No rows built — check file parsing above before writing.")
 
goalie_df = spark.createDataFrame(rows)
 
table_exists = spark.catalog.tableExists(TABLE)
 
if not table_exists:
    (
        goalie_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(TABLE)
    )
    print(f"Table created: {TABLE}")
    print(f"Rows written : {len(rows)}")
 
else:
    goalie_df.createOrReplaceTempView("new_goalie_logs")
 
    spark.sql(f"""
        MERGE INTO {TABLE} AS target
        USING new_goalie_logs AS source
        ON target.athlete_id = source.athlete_id
        WHEN MATCHED THEN
            UPDATE SET *
        WHEN NOT MATCHED THEN
            INSERT *
    """)
    print(f"Merged into existing table: {TABLE}")

In [0]:
# ── VALIDATE ─────────────────────────────────────────────────────────────────
 
result = spark.sql(f"""
    SELECT
        athlete_id,
        athlete_name,
        team_abbreviation,
        total_game_rows,
        playoff_game_rows,
        regular_game_rows,
        season,
        pulled_at,
        ingested_at
    FROM {TABLE}
    ORDER BY team_abbreviation, athlete_name
""")
 
total = result.count()
print(f"Total goalies in {TABLE}: {total}")
result.show(100, truncate=False)

In [0]:
# ── SANITY CHECKS ─────────────────────────────────────────────────────────────
 
checks = spark.sql(f"""
    SELECT
        COUNT(*)                                            AS total_goalies,
        COUNT(DISTINCT team_abbreviation)                   AS teams_represented,
        COUNT(CASE WHEN athlete_id IS NULL  THEN 1 END)    AS null_athlete_ids,
        COUNT(CASE WHEN athlete_name IS NULL THEN 1 END)   AS null_names,
        COUNT(CASE WHEN total_game_rows = 0  THEN 1 END)   AS goalies_no_games,
        COUNT(CASE WHEN playoff_game_rows > 0 THEN 1 END)  AS goalies_with_playoff_games,
        COUNT(CASE WHEN regular_game_rows > 0 THEN 1 END)  AS goalies_with_reg_games,
        SUM(total_game_rows)                                AS total_game_rows,
        SUM(playoff_game_rows)                              AS total_playoff_rows,
        SUM(regular_game_rows)                              AS total_regular_rows,
        MIN(pulled_at)                                      AS earliest_pull,
        MAX(pulled_at)                                      AS latest_pull
    FROM {TABLE}
""")
 
print("Sanity checks:")
checks.show(truncate=False)